In [ ]:
from jupyter_dash import JupyterDash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# =========================
# Load & Train Model (same logic)
# =========================
data = pd.read_csv("diabetes.csv")

X = data.drop("Outcome", axis=1)
y = data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = round(accuracy_score(y_test, y_pred) * 100, 2)

# =========================
# Dash App
# =========================
app = JupyterDash(__name__)

app.layout = html.Div([

    # Header
    html.Div([
        html.H1("🩺 Diabetes Prediction System", className="title"),
        html.P(f"Model Accuracy: {accuracy}%", className="subtitle")
    ], className="header"),

    # Form Container
    html.Div([

        html.Div([
            html.Label("Pregnancies"),
            dcc.Input(id="preg", type="number", min=0, max=20, value=1)
        ], className="input-box"),

        html.Div([
            html.Label("Glucose"),
            dcc.Input(id="glucose", type="number", min=0, max=200, value=120)
        ], className="input-box"),

        html.Div([
            html.Label("Blood Pressure"),
            dcc.Input(id="bp", type="number", min=0, max=150, value=70)
        ], className="input-box"),

        html.Div([
            html.Label("Skin Thickness"),
            dcc.Input(id="skin", type="number", min=0, max=100, value=20)
        ], className="input-box"),

        html.Div([
            html.Label("Insulin"),
            dcc.Input(id="insulin", type="number", min=0, max=900, value=80)
        ], className="input-box"),

        html.Div([
            html.Label("BMI"),
            dcc.Input(id="bmi", type="number", min=0, max=60, value=25)
        ], className="input-box"),

        html.Div([
            html.Label("Pedigree"),
            dcc.Input(id="pedigree", type="number", min=0, max=3, value=0.5)
        ], className="input-box"),

        html.Div([
            html.Label("Age"),
            dcc.Input(id="age", type="number", min=1, max=120, value=30)
        ], className="input-box"),

        html.Button("Predict", id="predict-btn", className="btn"),

        html.Div(id="result", className="result-box")

    ], className="form-container")

], className="main")

# =========================
# Callback (same logic)
# =========================
@app.callback(
    Output("result", "children"),
    Input("predict-btn", "n_clicks"),
    State("preg", "value"),
    State("glucose", "value"),
    State("bp", "value"),
    State("skin", "value"),
    State("insulin", "value"),
    State("bmi", "value"),
    State("pedigree", "value"),
    State("age", "value"),
)
def predict(n, preg, glucose, bp, skin, insulin, bmi, pedigree, age):

    if n is None:
        return ""

    input_data = np.array([[preg, glucose, bp, skin, insulin, bmi, pedigree, age]])
    prediction = model.predict(input_data)

    if prediction[0] == 1:
        return html.Div("⚠ High Risk of Diabetes", className="error")
    else:
        return html.Div("✅ No Diabetes Detected", className="success")


# =========================
# Run
# =========================
if __name__ == "__main__":
   app.run_server(mode='inline')
